## PHASE 2: STRUCTURED FEATURE ENGINEERING + NLP PIPELINE
**Duration:** 2-3 weeks | **Deliverable:** vectorized_features.csv + frequency_tables

### Step 2.1: Consolidate Text Fields
**Create unified text column per posting:**

```python
df['consolidated_text'] = (
    df['title'].fillna('') + " " +
    df['company_profile'].fillna('') + " " +
    df['description'].fillna('') + " " +
    df['requirements'].fillna('') + " " +
    df['benefits'].fillna('')
)
```

### Step 2.2: Text Normalization
Apply **in sequence** to `consolidated_text`:

1. **Lowercase:** Convert all to lowercase
2. **Remove URLs:** Replace `http://...` with placeholder `[URL]`
3. **Remove HTML/HTML entities:** Remove `&amp;`, `<br>`, etc.
4. **Remove extra whitespace:** Collapse multiple spaces
5. **Remove punctuation:** Keep hyphens (e.g., "work-from-home" → "work from home")
6. **Remove numbers:** Strip standalone digits (9, 2014, etc.)

**Checkpoint:** Inspect 5 random samples to ensure text is readable after normalization

### Step 2.3: Tokenization & Stop Words Removal
1. **Tokenize:** Split on whitespace
2. **Remove stop words:** Use NLTK English stop words (remove: "the", "and", "is", "a", "an", etc.)
3. **Filter:** Keep tokens with 3+ characters (removes noise like "mr", "vs")

**Checkpoint:** Print token count distribution (should average 50-100 tokens per posting)

### Step 2.4: Extract N-grams for Domain Relevance
**Critical for data mining!** Create bigrams (2-word phrases) and trigrams (3-word phrases):

**Why:** Single words like "wire", "transfer", "data", "entry" appear in legitimate jobs too. But **phrases** like "wire transfer", "upfront payment", "data entry" are MORE informative fraud signals.

```python
from sklearn.feature_extraction.text import TfidfVectorizer

# Bigrams only (2-word phrases)
bigram_vectorizer = TfidfVectorizer(ngram_range=(2, 2), max_features=100)
bigrams_matrix = bigram_vectorizer.fit_transform(df['cleaned_text'])

# Extract bigram names
bigram_names = bigram_vectorizer.get_feature_names_out()
print("Top 20 Bigrams:", bigram_names[:20])
```

### Step 2.5: TF-IDF Vectorization
**Transform cleaned text into numerical features:**

```python
# TF-IDF on cleaned tokens (unigrams)
tfidf_vectorizer = TfidfVectorizer(
    max_features=350,        # Limit to top 350 terms
    min_df=2,                # Term must appear in 2+ documents
    max_df=0.95,             # Ignore terms appearing in 95%+ of docs
    ngram_range=(1, 1)       # Single words only (already handled bigrams separately)
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])
```

**Output:** 
- TF-IDF sparse matrix (17,533 postings × 350 features)
- Feature names: array of top 350 most discriminative terms


In [1]:
import pandas as pd
import re
import html
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# Load preprocessed dataset
input_path = '../data/processed/cleaned_data.csv'
df = pd.read_csv(input_path)
print('Loaded shape:', df.shape)

Loaded shape: (17533, 17)


In [3]:
# Step 2.1: Consolidate text fields into one column
df['consolidated_text'] = (
    df['title'].fillna('') + ' ' +
    df['company_profile'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['requirements'].fillna('') + ' ' +
    df['benefits'].fillna('')
)

# Drop original text columns after consolidation
df = df.drop(columns=['title', 'company_profile', 'description', 'requirements', 'benefits'])
print('Step 2.1 complete')

Step 2.1 complete


In [4]:
# Step 2.2: Text normalization function
def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)

    # 1) Lowercase
    text = text.lower()

    # 2) Replace URLs with placeholder
    text = re.sub(r"https?://\S+|www\.\S+", ' [URL] ', text)

    # 3) Remove HTML tags/entities
    text = re.sub(r'<[^>]+>', ' ', text)
    text = html.unescape(text)

    # 4) Remove punctuation (keep hyphen temporarily)
    text = re.sub(r'[^\w\s-]', ' ', text)

    # 5) Convert hyphens to spaces
    text = text.replace('-', ' ')

    # 6) Remove standalone numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # Final whitespace cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
# Apply Step 2.2 normalization
df['normalized_text'] = df['consolidated_text'].apply(normalize_text)

# Checkpoint: inspect 5 random normalized samples
df['normalized_text'].sample(5, random_state=42).to_list()

['financial controller workable is a venture backed startup making cloud based recruitment software for fast growing companies around the world we re looking for people who want to change the way companies and people meet each other life at workableat workable we are creating an environment that has all the excitement and intellectual stimulation of a startup minus the fads and pretension we don t work hour weeks but we do work in an efficient and disciplined manner we don t have ninjas and rock stars we have people who are outstanding at what they do we don t think it s old fashioned to have a sensible business model and we enjoy working with smart people learn more about workable and our employee benefits we are looking for an experienced financial controller to undertake all aspects of financial management including corporate accounting regulatory and financial reporting budget and forecasts preparation as well as development of internal control policies and procedures manage all ac

In [6]:
# Step 2.3: Prepare NLTK stop words
nltk.download('stopwords', quiet=True)
english_stopwords = set(stopwords.words('english'))


def tokenize_and_filter(text: str):
    # 1) Tokenize on whitespace
    tokens = text.split()

    # 2) Remove stop words
    tokens = [t for t in tokens if t not in english_stopwords]

    # 3) Keep tokens with 3+ characters
    tokens = [t for t in tokens if len(t) >= 3]

    return tokens

In [7]:
# Apply Step 2.3 tokenization and filtering
df['tokens'] = df['normalized_text'].apply(tokenize_and_filter)
df['cleaned_text'] = df['tokens'].apply(lambda toks: ' '.join(toks))
df['token_count'] = df['tokens'].apply(len)

# Checkpoint: token count distribution
print(df['token_count'].describe())
print(f"Average tokens per posting: {df['token_count'].mean():.2f}")

count    17533.000000
mean       242.243712
std        132.303510
min          5.000000
25%        143.000000
50%        230.000000
75%        316.000000
max       1393.000000
Name: token_count, dtype: float64
Average tokens per posting: 242.24


In [11]:
df.head()

,location,department,salary_range,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,consolidated_text,normalized_text,tokens,cleaned_text,token_count
0,"US, NY, New York",Marketing,Not Specified,0,1,0,Other,Internship,Not Specified,Not Specified,Marketing,0,"Marketing Intern We're Food52, and we've creat...",marketing intern we re food52 and we ve create...,"[marketing, intern, food52, created, groundbre...",marketing intern food52 created groundbreaking...,251
1,"NZ, , Auckland",Success,Not Specified,0,1,0,Full-time,Not Applicable,Not Specified,Marketing and Advertising,Customer Service,0,Customer Service - Cloud Video Production 90 S...,customer service cloud video production second...,"[customer, service, cloud, video, production, ...",customer service cloud video production second...,521
2,"US, IA, Wever",Unknown,Not Specified,0,1,0,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified,0,Commissioning Machinery Assistant (CMA) Valor ...,commissioning machinery assistant cma valor se...,"[commissioning, machinery, assistant, cma, val...",commissioning machinery assistant cma valor se...,233
3,"US, DC, Washington",Sales,Not Specified,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,Account Executive - Washington DC Our passion ...,account executive washington dc our passion fo...,"[account, executive, washington, passion, impr...",account executive washington passion improving...,483
4,"US, FL, Fort Worth",Unknown,Not Specified,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,Bill Review Manager SpotSource Solutions LLC i...,bill review manager spotsource solutions llc i...,"[bill, review, manager, spotsource, solutions,...",bill review manager spotsource solutions llc g...,346


In [8]:
# Step 2.4: Extract bigrams and trigrams from cleaned text
bigram_vectorizer = TfidfVectorizer(ngram_range=(2, 2), max_features=100)
trigram_vectorizer = TfidfVectorizer(ngram_range=(3, 3), max_features=100)

bigrams_matrix = bigram_vectorizer.fit_transform(df['cleaned_text'])
trigrams_matrix = trigram_vectorizer.fit_transform(df['cleaned_text'])

frequency_table_bigrams = pd.DataFrame({
    'ngram': bigram_vectorizer.get_feature_names_out(),
    'score': bigrams_matrix.sum(axis=0).A1
}).sort_values('score', ascending=False).reset_index(drop=True)

frequency_table_trigrams = pd.DataFrame({
    'ngram': trigram_vectorizer.get_feature_names_out(),
    'score': trigrams_matrix.sum(axis=0).A1
}).sort_values('score', ascending=False).reset_index(drop=True)

frequency_table_bigrams.to_csv('../data/processed/frequency_table_bigrams.csv', index=False)
frequency_table_trigrams.to_csv('../data/processed/frequency_table_trigrams.csv', index=False)

print('Top 10 bigrams:')
print(frequency_table_bigrams.head(10))

Top 10 bigrams:
                  ngram        score
0      customer service  1221.919501
1             full time  1116.952757
2      years experience  1016.951671
3  communication skills   861.249977
4          social media   718.913192
5            fast paced   715.301276
6             long term   593.081113
7          fast growing   581.594035
8          ability work   533.255083
9          high quality   532.047710


In [9]:
# Step 2.5: TF-IDF vectorization on cleaned text (unigrams)
tfidf_vectorizer = TfidfVectorizer(
    max_features=350,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 1)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{term}' for term in tfidf_feature_names]
)

print('TF-IDF matrix shape:', tfidf_matrix.shape)

TF-IDF matrix shape: (17533, 350)


In [10]:
# Finalize outputs: remove intermediate text columns and export vectorized features
cols_to_remove = ['consolidated_text', 'normalized_text', 'tokens', 'token_count']
df_final = df.drop(columns=[c for c in cols_to_remove if c in df.columns])

vectorized_features = pd.concat([df_final.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
vectorized_features.to_csv('../data/processed/vectorized_features.csv', index=False)

print('Final feature table shape:', vectorized_features.shape)
print('Saved: ../data/processed/vectorized_features.csv')

Final feature table shape: (17533, 363)
Saved: ../data/processed/vectorized_features.csv
